# Posterior Calibration Check

Estimation error tells you how far the posterior mean is from the truth. Calibration asks a different question: does the posterior covariance describe uncertainty correctly?

For a calibrated Gaussian posterior `N(mu_T, Sigma_T)`, the true `theta` should behave like a draw from that posterior. This notebook checks marginal z-scores, Q-Q plots, credible-ellipsoid coverage, and completed-vs-dropout calibration.


## Setup


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm as norm_dist


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "synthetic.py").exists():
            return candidate
    raise RuntimeError("Could not find project root containing src/synthetic.py.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.belief import BeliefState
from src.metrics import calibration_result
from src.plots import (
    ACTIVE_BLACK,
    POSTERIOR_GREEN,
    PRIOR_BLUE,
    QUESTION_ORANGE,
    STRUCTURE_GRAY,
    apply_notebook_style,
    style_ax,
)
from src.policies import make_sensitive_constant_stay_prob
from src.simulate import simulate_population
from src.synthetic import generate_user_population, synthetic_item_bank

apply_notebook_style()
print(f"Using project root: {PROJECT_ROOT}")


## Simulation Configuration

Adjust these to match the regime you care about. The default policy set excludes `myopic_exact` because calibration runs can be slow at population scale.


In [ ]:
DIM = 6
N_ITEMS = 50
N_CATEGORIES = 4
SENSITIVE_FRACTION = 0.30
SENSITIVITY_ASSIGNMENT = "high_trait_tail"  # "random", "axis_aligned", or "high_trait_tail"
SENSITIVE_AXES = [2, 5]
P_DROPOUT = 0.10
N_USERS = 500
HORIZON = 20

SEED_BANK = 29
SEED_POP = 1
SEED_SIM = 2

POLICIES = [
    "fixed",
    "random",
    "surrogate_weighted",
]

POLICY_STYLE = {
    "fixed": {"color": STRUCTURE_GRAY, "ls": "--", "label": "Fixed"},
    "random": {"color": PRIOR_BLUE, "ls": "--", "label": "Random"},
    "myopic_exact": {"color": ACTIVE_BLACK, "ls": "-", "label": "Myopic (exact)"},
    "surrogate_unweighted": {"color": QUESTION_ORANGE, "ls": "-", "label": "Surrogate (unweighted)"},
    "surrogate_weighted": {"color": POSTERIOR_GREEN, "ls": "-", "label": "Surrogate (weighted)"},
}


## Simulate Policies


In [ ]:
item_bank = synthetic_item_bank(
    n_items=N_ITEMS,
    dim=DIM,
    n_categories=N_CATEGORIES,
    sensitive_fraction=SENSITIVE_FRACTION,
    sensitivity_assignment=SENSITIVITY_ASSIGNMENT,
    sensitive_axes=SENSITIVE_AXES,
    rng_seed=SEED_BANK,
)
theta_trues = generate_user_population(n_users=N_USERS, dim=DIM, rng_seed=SEED_POP)

stay_prob = make_sensitive_constant_stay_prob(p_stay_sensitive=1.0 - P_DROPOUT)
prior = BeliefState(mu=np.zeros(DIM), Sigma=np.eye(DIM))

policy_results = {}

for policy_index, policy in enumerate(POLICIES):
    print(f"  simulating {policy} ...", end=" ", flush=True)
    results = simulate_population(
        theta_trues=theta_trues,
        prior_belief=prior,
        item_bank=item_bank,
        horizon=HORIZON,
        strategy=policy,
        stay_prob_fn=stay_prob,
        rng=np.random.default_rng([SEED_SIM, policy_index]),
    )
    policy_results[policy] = results
    n_drop = sum(result.terminated_by_dropout for result in results)
    print(f"done  (dropout {n_drop}/{N_USERS} = {n_drop / N_USERS:.0%})")

print("\nAll simulations complete.")


## Calibration Objects


In [ ]:
calib = {}
for policy, results in policy_results.items():
    calib[policy] = calibration_result(
        results,
        theta_trues,
        filter_dropouts=False,
    )
    c = calib[policy]
    print(f"{policy:25s}  n={c.n_users}  dropouts excluded={c.n_dropouts_excluded}")


## 1. Marginal Z-Score Distribution

If calibrated, marginal z-scores should look approximately standard normal.


In [ ]:
n_pol = len(POLICIES)
fig, axes = plt.subplots(1, n_pol, figsize=(4.5 * n_pol, 4), sharey=True)
if n_pol == 1:
    axes = [axes]

z_grid = np.linspace(-4, 4, 300)
ideal = norm_dist.pdf(z_grid)

for ax, policy in zip(axes, POLICIES):
    c = calib[policy]
    style = POLICY_STYLE[policy]
    ax.hist(
        c.marginal_z_scores,
        bins=50,
        density=True,
        color=style["color"],
        alpha=0.70,
        edgecolor="white",
        linewidth=0.3,
    )
    ax.plot(z_grid, ideal, color=ACTIVE_BLACK, lw=1.5, label="N(0, 1)")
    ax.set_title(style["label"])
    ax.set_xlabel("marginal z-score")
    style_ax(ax, grid_axis="y")

axes[0].set_ylabel("density")
axes[-1].legend(frameon=True)
plt.tight_layout()
plt.show()


## 2. Q-Q Plot Of Marginal Z-Scores

If calibrated, points should lie near the diagonal. Tail deviations are especially important for credible intervals.


In [ ]:
fig, axes = plt.subplots(1, n_pol, figsize=(4.5 * n_pol, 4), sharey=True)
if n_pol == 1:
    axes = [axes]

for ax, policy in zip(axes, POLICIES):
    c = calib[policy]
    style = POLICY_STYLE[policy]
    z = np.sort(c.marginal_z_scores)
    p = (np.arange(1, len(z) + 1) - 0.5) / len(z)
    q = norm_dist.ppf(p)
    lim = max(3.5, float(np.nanmax(np.abs(z))))
    ax.scatter(q, z, s=8, color=style["color"], alpha=0.45, edgecolor="none")
    ax.plot([-lim, lim], [-lim, lim], color=ACTIVE_BLACK, lw=1.2, ls="--")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_title(style["label"])
    ax.set_xlabel("theoretical N(0, 1) quantile")
    style_ax(ax, grid_axis="both")

axes[0].set_ylabel("observed z-score quantile")
plt.tight_layout()
plt.show()


## 3. Credible-Ellipsoid Coverage Curve

The x-axis is nominal coverage. The y-axis is the empirical fraction of true users inside the corresponding posterior ellipsoid. The diagonal is perfect calibration.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], color=ACTIVE_BLACK, ls="--", lw=1.2, label="Perfect calibration")

for policy in POLICIES:
    c = calib[policy]
    style = POLICY_STYLE[policy]
    ax.plot(
        c.alphas,
        c.empirical_coverage,
        color=style["color"],
        ls=style["ls"],
        lw=1.8,
        label=style["label"],
    )

ax.set_title("Joint credible-ellipsoid coverage")
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
style_ax(ax, grid_axis="both")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


## 4. Per-Dimension 90% Marginal Coverage


In [ ]:
def per_dim_coverage_90(results, theta_trues_arr, alpha=0.90):
    z = norm_dist.ppf((1.0 + alpha) / 2.0)
    theta_arr = np.asarray(theta_trues_arr, dtype=float)
    dim = results[0].final_belief.dim
    coverages = []
    for dim_index in range(dim):
        inside = []
        for result, theta in zip(results, theta_arr):
            mu = result.final_belief.mu[dim_index]
            sigma = np.sqrt(result.final_belief.Sigma[dim_index, dim_index])
            inside.append(abs(theta[dim_index] - mu) <= z * sigma)
        coverages.append(float(np.mean(inside)))
    return coverages


fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(DIM)
width = 0.80 / max(len(POLICIES), 1)

for index, policy in enumerate(POLICIES):
    style = POLICY_STYLE[policy]
    offsets = x - 0.40 + width / 2.0 + index * width
    covs = per_dim_coverage_90(policy_results[policy], theta_trues)
    ax.bar(offsets, np.array(covs) * 100.0, width=width, color=style["color"], label=style["label"])

ax.axhline(90, color=ACTIVE_BLACK, ls="--", lw=1.2, label="Ideal 90%")
ax.set_title("Per-dimension 90% marginal coverage")
ax.set_xlabel("latent dimension")
ax.set_ylabel("coverage (%)")
ax.set_xticks(x)
ax.set_ylim(0, 105)
style_ax(ax, grid_axis="y")
ax.legend(frameon=True, ncol=2)
plt.tight_layout()
plt.show()


## 5. Completed Vs Dropout Calibration

This split shows whether early termination is associated with different calibration. The policy with the most dropouts is selected automatically.


In [ ]:
policy = max(POLICIES, key=lambda p: sum(result.terminated_by_dropout for result in policy_results[p]))
results_p = policy_results[policy]
n_drop = sum(result.terminated_by_dropout for result in results_p)
print(f"Using policy '{policy}' ({n_drop}/{N_USERS} dropouts)")

mask_complete = np.array([not result.terminated_by_dropout for result in results_p], dtype=bool)
mask_dropout = ~mask_complete

res_complete = [result for result, keep in zip(results_p, mask_complete) if keep]
res_dropout = [result for result, keep in zip(results_p, mask_dropout) if keep]
theta_complete = theta_trues[mask_complete]
theta_dropout = theta_trues[mask_dropout]

c_complete = calibration_result(res_complete, theta_complete) if res_complete else None
c_dropout = calibration_result(res_dropout, theta_dropout) if res_dropout else None

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], color=ACTIVE_BLACK, ls="--", lw=1.2, label="Perfect")
if c_complete is not None:
    ax.plot(c_complete.alphas, c_complete.empirical_coverage, color=POSTERIOR_GREEN, lw=1.8, label=f"Completed (n={c_complete.n_users})")
if c_dropout is not None:
    ax.plot(c_dropout.alphas, c_dropout.empirical_coverage, color=QUESTION_ORANGE, lw=1.8, ls=":", label=f"Dropped out (n={c_dropout.n_users})")

ax.set_title(f"Coverage: completed vs dropout ({POLICY_STYLE[policy]['label']})")
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
style_ax(ax, grid_axis="both")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()
